# Fine-tune EfficientNet on NEU surface-defect classification

Trains the **deep half** of the Surface Defect Inspector project on a GPU (this
runs in Colab because the build sandbox has no GPU and can't reach the dataset).
It produces three artifacts you drop into the repo's `models/` folder:

- `model.onnx` -- the fine-tuned EfficientNet, exported to ONNX so the app can serve it
  live with lightweight `onnxruntime` (no torch needed in production).
- `baseline_lbp_svm.joblib` -- the classical LBP+SVM baseline, trained on the **same split**
  so the comparison is fair.
- `results_summary.json` -- the measured numbers for both models.

**Runtime -> Change runtime type -> T4 GPU** before running. Then Run all.


In [ ]:
# EfficientNet + torchvision are preinstalled in Colab; add ONNX export + skimage.
!pip -q install onnx onnxscript scikit-image 2>/dev/null
import torch
print("CUDA available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Get the NEU surface-defect dataset

The **NEU-CLS** dataset is 1,800 grayscale images (200x200), 300 per class, across six
defect types. The most reliable source is Kaggle. Easiest path in Colab:

1. Get a Kaggle API token (kaggle.com -> Account -> Create New API Token -> `kaggle.json`).
2. Upload it when asked, or set `KAGGLE_USERNAME` / `KAGGLE_KEY`.

If you already have the data in Google Drive, skip this and set `DATA_ROOT` to its folder.
The loader below is tolerant of layout -- it finds images by the NEU 2-letter filename
prefix (`Cr/In/Pa/PS/RS/Sc`) **or** by class-named subfolders.

In [ ]:
import os, glob
DATA_ROOT = None

# --- Option A: Kaggle download ----------------------------------------------
try:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")
    print("Downloaded to:", DATA_ROOT)
except Exception as e:
    print("kagglehub download not available yet:", e)
    print("Provide the data another way and set DATA_ROOT to its folder.")

# --- Option B: point at a Google Drive / local folder -----------------------
# from google.colab import drive; drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/NEU-CLS'

assert DATA_ROOT and os.path.isdir(DATA_ROOT), "Set DATA_ROOT to the dataset folder."
imgs = [p for p in glob.glob(os.path.join(DATA_ROOT, '**', '*'), recursive=True)
        if p.lower().endswith(('.bmp', '.png', '.jpg', '.jpeg'))]
print("Found", len(imgs), "images")

In [ ]:
# Single source of truth for the six classes -- MUST match app/labels.py in the repo.
CLASS_NAMES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
NEU_PREFIX = {"cr": "crazing", "in": "inclusion", "pa": "patches",
              "ps": "pitted_surface", "rs": "rolled-in_scale", "sc": "scratches"}

def infer_label(path):
    parent = os.path.basename(os.path.dirname(path)).lower()
    if parent in CLASS_NAMES:
        return parent
    stem = os.path.basename(path).lower()
    for pre, cls in NEU_PREFIX.items():
        if stem.startswith(pre + "_") or (stem.startswith(pre) and stem[len(pre):len(pre)+1].isdigit()):
            return cls
    return None

items = [(p, infer_label(p)) for p in imgs]
items = [(p, l) for p, l in items if l in CLASS_NAMES]
from collections import Counter
print("Per-class counts:", Counter(l for _, l in items))
assert len(items) > 0, "No labeled images found -- check the dataset layout."
paths = [p for p, _ in items]; labels = [CLASS_NAMES.index(l) for _, l in items]

In [ ]:
# Fixed stratified split -- SEED=42, test_size=0.25 -- MUST match scripts/evaluate.py
from sklearn.model_selection import train_test_split
import numpy as np
SEED = 42
idx = np.arange(len(paths))
tr_idx, te_idx = train_test_split(idx, test_size=0.25, random_state=SEED, stratify=labels)
tr_idx, va_idx = train_test_split(tr_idx, test_size=0.2, random_state=SEED,
                                  stratify=[labels[i] for i in tr_idx])
print(f"train {len(tr_idx)}  val {len(va_idx)}  test {len(te_idx)}")

In [ ]:
# EfficientNet-B0 (ImageNet-pretrained), head replaced with a 6-class layer.
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

DEEP_SIZE = 224
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((DEEP_SIZE, DEEP_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15), transforms.ToTensor(), norm])
eval_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((DEEP_SIZE, DEEP_SIZE)),
    transforms.ToTensor(), norm])

class NEU(Dataset):
    def __init__(self, ids, tf): self.ids = ids; self.tf = tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        return self.tf(Image.open(paths[j]).convert("RGB")), labels[j]

dl_tr = DataLoader(NEU(tr_idx, train_tf), batch_size=32, shuffle=True, num_workers=2)
dl_va = DataLoader(NEU(va_idx, eval_tf), batch_size=64, num_workers=2)
dl_te = DataLoader(NEU(te_idx, eval_tf), batch_size=64, num_workers=2)

dev = "cuda" if torch.cuda.is_available() else "cpu"
net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
net.classifier[1] = nn.Linear(net.classifier[1].in_features, len(CLASS_NAMES))
net = net.to(dev)

In [ ]:
# Fine-tune (short -- transfer learning converges fast on ~1.3k images).
opt = torch.optim.Adam(net.parameters(), lr=3e-4)
lossf = nn.CrossEntropyLoss()
EPOCHS = 8
for ep in range(EPOCHS):
    net.train()
    for x, y in dl_tr:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad(); loss = lossf(net(x), y); loss.backward(); opt.step()
    net.eval(); ok = tot = 0
    with torch.no_grad():
        for x, y in dl_va:
            p = net(x.to(dev)).argmax(1).cpu()
            ok += (p == y).sum().item(); tot += len(y)
    print(f"epoch {ep+1}/{EPOCHS}  val_acc={ok/tot:.4f}")

In [ ]:
# Evaluate the CNN on the held-out TEST split.
from sklearn.metrics import accuracy_score, f1_score
net.eval(); yt, yp = [], []
with torch.no_grad():
    for x, y in dl_te:
        yp += net(x.to(dev)).argmax(1).cpu().tolist(); yt += y.tolist()
order = list(range(len(CLASS_NAMES)))
cnn = {"accuracy": float(accuracy_score(yt, yp)),
       "macro_f1": float(f1_score(yt, yp, average="macro")),
       "per_class_f1": {CLASS_NAMES[c]: float(f) for c, f in
                        zip(order, f1_score(yt, yp, labels=order, average=None))}}
print("CNN  acc", round(cnn["accuracy"],4), " macroF1", round(cnn["macro_f1"],4))

In [ ]:
# Classical LBP+SVM baseline on the SAME split -- code identical to app/baseline.py
import cv2
from skimage.feature import local_binary_pattern
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

BASE_SIZE = 200
_LBP = [(1, 8), (3, 24)]
def lbp_features(gray):
    if gray.shape != (BASE_SIZE, BASE_SIZE): gray = cv2.resize(gray, (BASE_SIZE, BASE_SIZE))
    feats = []
    for r, p in _LBP:
        codes = local_binary_pattern(gray, p, r, method="uniform")
        h, _ = np.histogram(codes.ravel(), bins=p+2, range=(0, p+2)); h = h.astype(float); h /= h.sum()+1e-7
        feats.append(h)
    return np.concatenate(feats)

def feats_for(ids):
    return np.array([lbp_features(cv2.imread(paths[i], cv2.IMREAD_GRAYSCALE)) for i in ids])
Xtr, ytr = feats_for(tr_idx), np.array([labels[i] for i in tr_idx])
Xte, yte = feats_for(te_idx), np.array([labels[i] for i in te_idx])
pipe = Pipeline([("scale", StandardScaler()),
                 ("clf", SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42))])
pipe.fit(Xtr, ytr); bp = pipe.predict(Xte)
base = {"accuracy": float(accuracy_score(yte, bp)),
        "macro_f1": float(f1_score(yte, bp, average="macro")),
        "per_class_f1": {CLASS_NAMES[c]: float(f) for c, f in
                         zip(order, f1_score(yte, bp, labels=order, average=None))},
        "confusion_matrix": confusion_matrix(yte, bp, labels=order).tolist(),
        "n_train": int(len(ytr)), "n_test": int(len(yte))}
print("Baseline  acc", round(base["accuracy"],4), " macroF1", round(base["macro_f1"],4))

In [ ]:
# Honest side-by-side comparison.
print("="*60)
print(f"{'Metric':<14}{'Baseline':>12}{'EfficientNet':>15}{'Delta':>10}")
print("="*60)
print(f"{'Accuracy':<14}{base['accuracy']:>12.4f}{cnn['accuracy']:>15.4f}{cnn['accuracy']-base['accuracy']:>+10.4f}")
print(f"{'Macro F1':<14}{base['macro_f1']:>12.4f}{cnn['macro_f1']:>15.4f}{cnn['macro_f1']-base['macro_f1']:>+10.4f}")
for c in CLASS_NAMES:
    print(f"  {c:16} baseline={base['per_class_f1'][c]:.4f}  cnn={cnn['per_class_f1'][c]:.4f}")
print("="*60)
verdict = "beat" if cnn['accuracy'] > base['accuracy'] else "did NOT beat"
print(f"Fine-tuned EfficientNet {verdict} the classical baseline -- measured, not assumed.")

In [ ]:
# Export the CNN to ONNX (what the app serves live) + save the baseline joblib.
import joblib, json
net.eval()
dummy = torch.randn(1, 3, DEEP_SIZE, DEEP_SIZE, device=dev)
torch.onnx.export(net, dummy, "model.onnx", input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
                  opset_version=17, dynamo=False)
print("Wrote model.onnx", round(os.path.getsize("model.onnx")/1e6, 1), "MB")

joblib.dump({"pipeline": pipe, "metrics": base, "class_names": CLASS_NAMES}, "baseline_lbp_svm.joblib")
json.dump({"baseline": base, "efficientnet": cnn}, open("results_summary.json", "w"), indent=2)
print("Wrote baseline_lbp_svm.joblib and results_summary.json")

In [ ]:
# Download all three artifacts. Drop model.onnx and baseline_lbp_svm.joblib into the
# repo's models/ folder; results_summary.json is just a record of the numbers.
from google.colab import files
for f in ["model.onnx", "baseline_lbp_svm.joblib", "results_summary.json"]:
    files.download(f)